Alumnos: Daniel Sánchez Piqueras (100526354), Miguel Trimiño Girón (100522142)

# Notebook 2 — Carga del Modelo Final y Predicciones de Competición

Este notebook carga el modelo final entrenado (`modelo_final.joblib`) y lo utiliza
para generar predicciones sobre el dataset de competición (`bank_competition.pkl`).
El propósito es únicamente el uso del modelo, no su entrenamiento ni evaluación.

# Importamos los datos necesarios para este notebook del notebook2

In [ ]:
import joblib
import pandas as pd
import numpy as np

def limpiar_nulos(X):
    return pd.DataFrame(X).replace([None, 'None', 'unknown', 'Unknown'], np.nan)
# Cargamos el modelo final
modelo_final = joblib.load('modelo_final.joblib')
print("Modelo cargado correctamente.")
print(f"Tipo de modelo: {type(modelo_final.named_steps['clf']).__name__}")

# Cargamos el dataset de competición
ds_competicion = pd.read_pickle('bank_competition.pkl')
print(f"Dataset de competición: {ds_competicion.shape}")
ds_competicion.head(3)

# Preproceso de pdays
ds_competicion['contactado_antes'] = (ds_competicion['pdays'] != -1).astype(int)
ds_competicion['pdays'] = ds_competicion['pdays'].replace(-1, 0)
print("Preproceso de pdays aplicado.")

# Generamos las predicciones con este bloque
preds_raw = modelo_final.predict(ds_competicion)
pred_labels = ['yes' if p == 1 else 'no' for p in preds_raw]

pred_df = pd.DataFrame({'deposit': pred_labels})
pred_df.to_csv('predicciones.csv', index=False)

print(f"Total predicciones generadas: {len(pred_df)}")
print("Distribución:")
print(pred_df['deposit'].value_counts())
pred_df.head(10)

Modelo cargado correctamente.
Tipo de modelo: SVC
Dataset de competición: (162, 16)
Preproceso de pdays aplicado.
Total predicciones generadas: 162
Distribución:
deposit
no    162
Name: count, dtype: int64


,deposit
0,no
1,no
2,no
3,no
4,no
5,no
6,no
7,no
8,no
9,no


## Verificación: dos instancias nuevas

Comprobamos que el modelo cargado produce las mismas predicciones que se mostrarán
en la aplicación Streamlit para estas dos instancias concretas.
Estos resultados deben coincidir con los de la app al introducir los mismos valores.

In [ ]:
Ejemplos de predicciones para luego comparar con el Streamlit y sacar las capturas
instancias_nuevas = pd.DataFrame([
    {
        'age': 42, 'job': 'management', 'marital': 'married', 'education': 'tertiary',
        'default': 'no', 'balance': 2500, 'housing': 'yes', 'loan': 'no',
        'contact': 'cellular', 'day': 15, 'month': 'may', 'duration': 320,
        'campaign': 2, 'pdays': 100, 'previous': 1, 'poutcome': 'success',
        'contactado_antes': 1
    },
    {
        'age': 28, 'job': 'blue-collar', 'marital': 'single', 'education': 'secondary',
        'default': 'no', 'balance': 150, 'housing': 'yes', 'loan': 'yes',
        'contact': 'unknown', 'day': 8, 'month': 'jun', 'duration': 80,
        'campaign': 5, 'pdays': 0, 'previous': 0, 'poutcome': 'failure',
        'contactado_antes': 0
    }
])

preds = modelo_final.predict(instancias_nuevas)
probas = modelo_final.predict_proba(instancias_nuevas)[:, 1]

print("Predicciones para comparar con Streamlit:")
for i, (pred, proba) in enumerate(zip(preds, probas)):
    print(f"  Instancia {i+1}: {pred}  (probabilidad no suscripción: {1-proba:.4f}) (probabilidad suscripción: {proba:.4f})")

Predicciones para comparar con Streamlit:
  Instancia 1: yes  (probabilidad no suscripción: 0.1566) (probabilidad suscripción: 0.8434)
  Instancia 2: no  (probabilidad no suscripción: 0.9180) (probabilidad suscripción: 0.0820)
